In [0]:
import uuid

In [0]:
accounts_data = [
    ("A100", "C001", "ACTIVE", 5000),
    ("A101", "C002", "ACTIVE", 8000),
    ("A102", "C003", "INACTIVE", 10000)
]

accounts_schema = StructType([
    StructField("account_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("status", StringType()),
    StructField("credit_limit", IntegerType())
])

df_accounts = spark.createDataFrame(accounts_data, accounts_schema)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Row

transactions_data = [
    ("T001", "A100", 200.50, "2024-03-01"),
    ("T002", "A100", -50.00, "2024-03-02"),
    ("T003", "A101", 500.00, "2024-03-03"),
    ("T003", "A101", 500.00, "2024-03-03"),  # duplicate
    ("T004", "A102", 1000.00, "2024-03-05"),
]

transactions_schema = StructType([
    StructField("txn_id", StringType()),
    StructField("account_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("txn_date", StringType())
])

df_source = spark.createDataFrame(transactions_data, transactions_schema)
df_source.show()

+------+----------+------+----------+
|txn_id|account_id|amount|  txn_date|
+------+----------+------+----------+
|  T001|      A100| 200.5|2024-03-01|
|  T002|      A100| -50.0|2024-03-02|
|  T003|      A101| 500.0|2024-03-03|
|  T003|      A101| 500.0|2024-03-03|
|  T004|      A102|1000.0|2024-03-05|
+------+----------+------+----------+



In [0]:
# batch_id = "2024-03"
batch_id = str(uuid.uuid4())
batch_id

'172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d'

In [0]:
df_bronze = df_source \
    .withColumn("batch_id", lit(batch_id)) \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source_file", lit("card_txn_202403.csv")) \
    .withColumn(
        "record_hash",
        sha2(concat_ws("||", *df_source.columns), 256)
    )

df_bronze.display()

txn_id,account_id,amount,txn_date,batch_id,ingestion_ts,source_file,record_hash
T001,A100,200.5,2024-03-01,172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d,2026-02-15T19:12:52.98315Z,card_txn_202403.csv,5876d064528687dcd5edf9a5315decd2817baf1a8782476e0a6edb2ab650ef93
T002,A100,-50.0,2024-03-02,172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d,2026-02-15T19:12:52.98315Z,card_txn_202403.csv,7b8830a3426dda2a136a58b5cc449edf5a915131e4ca9161b42b0bfbae30f689
T003,A101,500.0,2024-03-03,172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d,2026-02-15T19:12:52.98315Z,card_txn_202403.csv,fdcdc1883774b1be0995df00d1b450689691c61bc8059c8fca940cb64e7198e7
T003,A101,500.0,2024-03-03,172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d,2026-02-15T19:12:52.98315Z,card_txn_202403.csv,fdcdc1883774b1be0995df00d1b450689691c61bc8059c8fca940cb64e7198e7
T004,A102,1000.0,2024-03-05,172a9ee4-87f8-49e2-9b3f-9a2df3c08c1d,2026-02-15T19:12:52.98315Z,card_txn_202403.csv,e9e1e30802a314723c3dba704d33c1a1636fe441d0288859199992bbfa3ebb95


In [0]:
df_silver_txn = df_bronze.dropDuplicates(["txn_id"])
df_silver_txn = df_silver_txn \
    .withColumn("txn_date", to_date("txn_date"))

In [0]:
df_balance = df_silver_txn.groupBy("account_id") \
    .agg(sum("amount").alias("total_balance"))
df_balance.show()

+----------+-------------+
|account_id|total_balance|
+----------+-------------+
|      A102|       1000.0|
|      A100|        150.5|
|      A101|        500.0|
+----------+-------------+



In [0]:
df_accounts.display()

account_id,customer_id,status,credit_limit
A100,C001,ACTIVE,5000
A101,C002,ACTIVE,8000
A102,C003,INACTIVE,10000


In [0]:
df_silver = df_balance.join(df_accounts, "account_id", "left")
df_silver.display()

account_id,total_balance,customer_id,status,credit_limit
A102,1000.0,C003,INACTIVE,10000
A100,150.5,C001,ACTIVE,5000
A101,500.0,C002,ACTIVE,8000


Business rule check:

In [0]:
df_silver = df_silver.withColumn(
    "rule_flag",
    when(
        (col("status") == "INACTIVE") & (col("total_balance") != 0),
        "REJECT"
    ).otherwise("PASS")
)
df_silver.display()

account_id,total_balance,customer_id,status,credit_limit,rule_flag
A102,1000.0,C003,INACTIVE,10000,REJECT
A100,150.5,C001,ACTIVE,5000,PASS
A101,500.0,C002,ACTIVE,8000,PASS


GOLD（Risk Layer）

In [0]:
df_reject = df_silver.filter(col("rule_flag") == "REJECT")
df_valid  = df_silver.filter(col("rule_flag") == "PASS")

In [0]:
df_gold = df_valid.withColumn(
    "utilization_ratio",
    col("total_balance") / col("credit_limit")
)
df_gold.display()

account_id,total_balance,customer_id,status,credit_limit,rule_flag,utilization_ratio
A100,150.5,C001,ACTIVE,5000,PASS,0.0301
A101,500.0,C002,ACTIVE,8000,PASS,0.0625


In [0]:
df_gold = df_gold.withColumn(
    "risk_segment",
    when(col("utilization_ratio") < 0.3, "LOW")
    .when(col("utilization_ratio") < 0.7, "MEDIUM")
    .otherwise("HIGH")
)

df_gold.show()

+----------+-------------+-----------+------+------------+---------+-----------------+------------+
|account_id|total_balance|customer_id|status|credit_limit|rule_flag|utilization_ratio|risk_segment|
+----------+-------------+-----------+------+------------+---------+-----------------+------------+
|      A100|        150.5|       C001|ACTIVE|        5000|     PASS|           0.0301|         LOW|
|      A101|        500.0|       C002|ACTIVE|        8000|     PASS|           0.0625|         LOW|
+----------+-------------+-----------+------+------------+---------+-----------------+------------+

